# Train and evaluate the translator on Google Colab

Before running anything: **Runtime → Change runtime type → T4 GPU**.

Steps: check the GPU → get the code → (optional) quick trial → full training → evaluation → look at the results.
Everything is saved to your Google Drive, so nothing is lost if Colab disconnects.

In [ ]:
!nvidia-smi

## 1. Get the code
Put your own GitHub repo address below (the repo that contains this project).

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/YOUR-REPO"   # <-- change this

!git clone $REPO_URL /content/nmt
%cd /content/nmt
!pip install -q -r requirements.txt

## 2. Connect Google Drive
The trained model and results are stored here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/nmt-translator'

## 3. Quick trial (a few minutes)
Trains on only 400 sentences, just to prove everything works before the long run. You can skip it.

In [ ]:
!python train.py --epochs 1 --max_train_samples 400 --eval_every 25 --val_samples 50 --output_dir /content/trial

## 4. Full training
Defaults: mT5-base, 2 epochs, batch size 8, sentences up to 128 tokens.
If you get a *CUDA out of memory* error, add `--batch_size 4`.
Watch the `min elapsed` in the log to see how long it will take.

In [ ]:
!python train.py --output_dir $OUT/model

## 5. Evaluate on the test sentences
Scores all 6 directions (en/ja/zh to each other) with BLEU and chrF. Add `--max_samples 200` for a faster, rougher check.

In [ ]:
!python evaluate.py --model_dir $OUT/model --out_dir $OUT/eval

## 6. Look at the results

In [ ]:
print(open(f'{OUT}/eval/results.md', encoding='utf-8').read())

from IPython.display import Image, display
display(Image(f'{OUT}/model/loss_curve.png'))

# a few example translations: direction, source, reference, model output
for line in open(f'{OUT}/eval/samples.tsv', encoding='utf-8').read().splitlines()[1:6]:
    print(line.replace('\t', '\n    '), '\n')

## 7. Try your own sentence
(The model folder in your Drive is what you later copy to your laptop as `outputs/mt5-translator`.)

In [ ]:
from core import load_model_and_tokenizer, translate
tokenizer, model = load_model_and_tokenizer(f'{OUT}/model')
for lang in ('ja', 'zh', 'en'):
    print(lang, '->', translate('What is your name?', lang, tokenizer, model))